# Exercice 2 : Transfer Learning — Chat vs Chien
## TP2 — Reseaux de Neurones Convolutifs (PyTorch)

---

**Objectif :** Classifier des images de chats et chiens en utilisant le Transfer Learning.

**On va comparer 4 modeles pre-entraines :**
1. **MobileNetV2** — leger, rapide
2. **VGG16** — classique, lourd
3. **ResNet50** — avec connexions residuelles
4. **EfficientNetB0** — optimise, performant

**Pourquoi Transfer Learning ?**
Entrainer un CNN from scratch necessite beaucoup de donnees et de temps.
Le Transfer Learning reutilise un modele deja entraine sur ImageNet (14 millions d'images).

---

## Etape 1 — Activer le GPU

**Runtime > Change runtime type > GPU**

In [ ]:
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## Etape 2 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, Subset
from torchvision.datasets import ImageFolder
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

---
## Etape 3 — Charger le dataset Cats vs Dogs

**Dataset :**
- 25 000 images couleur
- 2 classes : Chat (0) / Chien (1)
- Tailles variees -> on va toutes les redimensionner en 224x224

In [ ]:
import zipfile
import urllib.request

url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

if not os.path.exists('PetImages'):
    print("Telechargement du dataset...")
    urllib.request.urlretrieve(url, 'cats_dogs.zip')
    with zipfile.ZipFile('cats_dogs.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    os.remove('cats_dogs.zip')
    print("Termine !")
else:
    print("Dataset deja present.")

In [ ]:
# Normalisation pour ImageNet (pour les modeles pre-entraines)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Dataset avec filtrage des images corrompues
class CleanImageFolder(ImageFolder):
    def __getitem__(self, index):
        try:
            return super().__getitem__(index)
        except (UnidentifiedImageError, OSError):
            blank = Image.new('RGB', (224, 224))
            return transform(blank), -1

# Charger le dataset
full_dataset = CleanImageFolder(root='PetImages', transform=transform)

# Filtrer les images corrompues (label -1)
valid_indices = [i for i in range(len(full_dataset)) if full_dataset[i][1] != -1]
print(f"Images valides : {len(valid_indices)} / {len(full_dataset)}")

clean_dataset = Subset(full_dataset, valid_indices)

print(f"Nombre total d'images : {len(clean_dataset)}")
print(f"Classes : ['Cat', 'Dog']")

In [ ]:
# Separer train (80%), validation (10%), test (10%)
total = len(clean_dataset)
train_size = int(0.8 * total)
val_size = int(0.1 * total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    clean_dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train : {train_size} images")
print(f"Validation : {val_size} images")
print(f"Test : {test_size} images")

In [ ]:
# Afficher quelques images
mean = torch.tensor([0.485, 0.456, 0.406])
std = torch.tensor([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    ax = axes[i // 5, i % 5]
    img, label = clean_dataset[i]
    img = img * std[:, None, None] + mean[:, None, None]
    img = torch.clamp(img, 0, 1)
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title('Chien' if label == 1 else 'Chat')
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Etape 4 — Data Augmentation

In [ ]:
# Augmentation des donnees
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Appliquer l'augmentation au train
full_dataset_aug = CleanImageFolder(root='PetImages', transform=train_transform)
valid_indices_aug = [i for i in range(len(full_dataset_aug)) if full_dataset_aug[i][1] != -1]
clean_dataset_aug = Subset(full_dataset_aug, valid_indices_aug)
train_dataset_aug, _, _ = random_split(clean_dataset_aug, [train_size, val_size, test_size])
train_loader = DataLoader(train_dataset_aug, batch_size=32, shuffle=True, num_workers=2)

print("Data Augmentation appliquee !")

---
## Etape 5 — Creer une fonction pour construire un modele

**Principe du Transfer Learning :**
1. Charger le modele pre-entraine (sans la couche de classification)
2. **Geler** les couches convolutionnelles (on ne les modifie pas)
3. Ajouter nos propres couches pour la classification binaire

In [ ]:
import torchvision.models as models

def create_model(model_name):
    """Construit un modele avec Transfer Learning"""

    # 1. Charger le modele pre-entraine sur ImageNet
    if model_name == "MobileNetV2":
        model = models.mobilenet_v2(weights='IMAGENET1K_V1')
        num_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 1)
        )

    elif model_name == "VGG16":
        model = models.vgg16(weights='IMAGENET1K_V1')
        num_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(num_features, 1)

    elif model_name == "ResNet50":
        model = models.resnet50(weights='IMAGENET1K_V1')
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, 1)

    elif model_name == "EfficientNetB0":
        model = models.efficientnet_b0(weights='IMAGENET1K_V1')
        num_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 1)
        )

    # 2. Geler les couches du modele de base
    for param in model.parameters():
        param.requires_grad = False

    # 3. Debloquer la derniere couche (pour l'entrainer)
    for param in model.parameters():
        if param.requires_grad == False:
            param.requires_grad = True
            break

    return model.to(device)

print("Fonction creee !")

In [ ]:
# Tester avec MobileNetV2
test_model = create_model("MobileNetV2")
print(test_model)

---
## Etape 6 — Entrainer et evaluer chaque modele

Pour chaque modele, on fait :
1. Creer le modele
2. L'entrainer
3. Evaluer sur le jeu de test
4. Sauvegarder les resultats

In [ ]:
def train_model(name, model, epochs=10, lr=0.001):
    """Entraine un modele et renvoie l'historique + l'accuracy"""

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_acc_history = []
    val_acc_history = []

    best_val_acc = 0.0
    patience = 3
    patience_counter = 0

    for epoch in range(epochs):
        # --- Entrainement ---
        model.train()
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device).float()

            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = correct / total

        # --- Validation ---
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device).float()
                outputs = model(images).squeeze()
                predicted = (torch.sigmoid(outputs) > 0.5).float()
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total

        train_acc_history.append(train_acc)
        val_acc_history.append(val_acc)

        print(f"  Epoch {epoch+1}/{epochs} — Train: {train_acc:.2%} — Val: {val_acc:.2%}")

        # Early Stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping a l'epoch {epoch+1}")
                break

    # Evaluation sur le test set
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = model(images).squeeze()
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_acc = correct / total
    print(f"  Test accuracy {name} : {test_acc:.2%}\n")

    return train_acc_history, val_acc_history, test_acc

print("Fonction d'entrainement prete !")

In [ ]:
# Entrainer les 4 modeles un par un
MODELS = ["MobileNetV2", "VGG16", "ResNet50", "EfficientNetB0"]

results = {}
histories = {}

for name in MODELS:
    print("=" * 50)
    print(f"Entrainement : {name}")
    print("=" * 50)

    model = create_model(name)
    train_hist, val_hist, accuracy = train_model(name, model)

    histories[name] = {"train": train_hist, "val": val_hist}
    results[name] = accuracy

---
## Etape 7 — Comparaison des resultats

In [ ]:
# Tableau comparatif
print("Tableau comparatif :")
print("-" * 30)
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:20s} : {acc:.2%}")

In [ ]:
# Graphique en barres
noms = list(results.keys())
accs = list(results.values())

plt.figure(figsize=(10, 6))
bars = plt.bar(noms, accs, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'], width=0.5)
plt.ylim(0.8, 1.0)
plt.ylabel('Accuracy')
plt.title('Comparaison des 4 modeles (Transfer Learning)', fontsize=16)

for bar, a in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{a:.2%}', ha='center', fontsize=13, fontweight='bold')

plt.grid(True, axis='y', alpha=0.3)
plt.show()

---
## Etape 8 — Courbes d'apprentissage

In [ ]:
# Courbes d'Accuracy
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, name in enumerate(histories.keys()):
    row = i // 2
    col = i % 2

    axes[row, col].plot(histories[name]['train'], label='Train', linewidth=2)
    axes[row, col].plot(histories[name]['val'], label='Validation', linewidth=2)
    axes[row, col].set_title(f'{name} — Accuracy', fontsize=14)
    axes[row, col].set_xlabel('Epoch')
    axes[row, col].set_ylabel('Accuracy')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Etape 9 — Synthese

**Quel modele choisir ?**

| Modele | Params | Avantage | Inconvenient |
|--------|--------|----------|---------------|
| MobileNetV2 | ~3.4M | Rapide, leger | Moins precis |
| VGG16 | ~138M | Classique, bien compris | Tres lourd |
| ResNet50 | ~25.6M | Bonne precision | Plus lent |
| EfficientNetB0 | ~5.3M | Meilleur compromis | Nouveau, moins connu |

In [ ]:
# Resume final
print("=" * 50)
print("RESUME FINAL")
print("=" * 50)
print(f"Dataset : Cats vs Dogs ({len(clean_dataset)} images)")
print(f"Methode : Transfer Learning (PyTorch)")
print(f"Epochs : 10 (avec Early Stopping)")
print()

for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:20s} : {acc:.2%}")

print()
meilleur = max(results, key=results.get)
print(f"Meilleur modele : {meilleur} ({results[meilleur]:.2%})")

---
## Resume general

| Concept | Description |
|---------|-------------|
| **Transfer Learning** | Reutiliser un modele pre-entraine sur ImageNet |
| **Data Augmentation** | Augmenter la diversite du dataset (flip, rotation, zoom) |
| **Early Stopping** | Arreter l'entrainement si le modele ne s'ameliore plus |
| **BCEWithLogitsLoss** | Fonction de perte pour la classification binaire |
| **requires_grad** | Geler/debloquer les couches dans PyTorch |

**Pourquoi le Transfer Learning est efficace ?**
- Les premieres couches des CNN apprennent des features universelles (contours, textures)
- On reutilise ces features et on n'entraine que le classifieur
- Resultat : entrainement rapide et bonne precision